# Day 1 -- From pixels to vision transformers

Today we build **five models** on the same medical images and watch each one beat
the last: logistic regression -> MLP -> CNN -> ResNet -> Vision Transformer.

The dataset is **APTOS-2019**: color photos of the back of the eye (fundus), graded
0-4 for diabetic retinopathy. This was the first AI screening tool deployed at scale
in real clinics.

**How this works:** fill in the `# TODO` lines. Stuck on one? Ask Claude -- then make
sure you understand what it gives you before moving on. The point isn't to finish
fastest, it's to understand why each model does better than the one before.

In [ ]:
import sys
sys.path.insert(0, "../_shared")
import colab_setup
colab_setup.ensure()
colab_setup.gpu_check()

In [ ]:
import torch
sys.path.insert(0, "..")
from day1_ladder import common

device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("device:", device)
results = {}   # we'll collect each model's validation accuracy here

## Step 0 -- What is an image, really?

Before we model anything: an image is just a grid of numbers. A color image is
*three* grids stacked (red, green, blue). Let's look.

In [ ]:
train_loader, val_loader = common.get_loaders(size=224, batch_size=32)
images, labels = next(iter(train_loader))
print("one batch:", images.shape, "  <- (batch, channels, height, width)")

common.show_rgb_split(images[0])
common.show_pixel_histogram(images[0])

Augmentation = small random changes (flip, rotate, brighten) so the model sees more
variety and doesn't memorize. Here's the same eye, augmented a few ways:

In [ ]:
import torchvision.transforms as T
# turn one (de-normalized) training image back into a PIL image to show augmentations
sample_pil = T.ToPILImage()(common._denorm(images[0]))
common.show_augmentations(sample_pil)

## Step 1 -- Logistic regression

The simplest classifier there is. We flatten each image into one long row of numbers
and fit a linear model. We use small 64x64 images here so it runs fast.

**Predict first:** there are 5 grades. Random guessing would be ~20%. How well do you
think a linear model on raw pixels will do? Write your guess.

In [ ]:
from sklearn.linear_model import LogisticRegression

tr64, va64 = common.get_loaders(size=64, batch_size=64)
Xtr, ytr = common.flatten_for_classical(tr64)
Xva, yva = common.flatten_for_classical(va64)
print("flattened features per image:", Xtr.shape[1])

# TODO: make a LogisticRegression with max_iter=500 and solver='saga'
# TODO: fit the classifier on the training features Xtr, ytr

acc = (clf.predict(Xva) == yva).mean()
results["logreg"] = acc
print(f"logistic regression val accuracy: {acc:.3f}")

*How close was your guess?* Logistic regression treats every pixel as independent and
can only draw straight-line boundaries. It has no idea that neighboring pixels form
shapes. Hold that thought.

## Step 2 -- Multi-layer perceptron (a small neural net)

Same flattened pixels, but now we stack layers with non-linearities so the model can
bend its decision boundaries. Still no notion of space, though.

In [ ]:
# TODO: build an MLP with common.make_mlp; in_features is 3*64*64 (the flattened size)
# TODO: train it: common.train_model(mlp, tr64, va64, epochs=5, lr=1e-3, device=device)
results["mlp"] = history[-1][1]
print(f"mlp val accuracy: {history[-1][1]:.3f}")

*Did it beat logreg by much?* Usually only a little. More layers help, but the model
still can't see that pixels next to each other belong together. That's the next jump.

## Step 3 -- Convolutional neural network (from scratch)

A CNN slides small filters across the image, so it *does* understand spatial structure
-- edges, blobs, textures. Now we use the full 224x224 images.

**Predict:** how much will accuracy jump versus the MLP?

In [ ]:
tr224, va224 = common.get_loaders(size=224, batch_size=32)

# TODO: build the CNN with common.make_small_cnn()
# TODO: train it for 5 epochs at lr=1e-3 on the 224px loaders
results["cnn"] = history[-1][1]
print(f"cnn val accuracy: {history[-1][1]:.3f}")

common.show_first_layer_filters(cnn)

Look at the first-layer filters above. Even this tiny CNN learned little edge and
color detectors on its own. *That* is what convolutions buy you over a plain MLP.

## Step 4 -- ResNet50 (transfer learning)

Instead of learning from scratch, we take a 50-layer network already trained on a
million everyday photos (ImageNet) and reuse its "vision." We freeze it and train just
a new final layer for our 5 eye grades. This is the single biggest practical trick in
modern computer vision.

In [ ]:
# TODO: build a pretrained ResNet50 with common.make_resnet50(pretrained=True)
# TODO: finetune for 3 epochs at lr=1e-3 on the 224px loaders
results["resnet"] = history[-1][1]
print(f"resnet val accuracy: {history[-1][1]:.3f}")

# where is it looking?
imgs, _ = next(iter(va224))
cam, predicted = common.gradcam(resnet, imgs[0], device=device)
print("Grad-CAM heatmap shape:", cam.shape, "predicted grade:", predicted)

*Why did pretraining help so much with so little training?* The network already knew
how to see edges, textures, and shapes. We only had to teach it which combinations
mean "diabetic retinopathy."

## Step 5 -- Vision Transformer

A ViT chops the image into patches, turns each patch into a vector, and lets the
patches "pay attention" to each other. Same idea as the language models you've used --
just patches instead of words. (More on that tomorrow.)

In [ ]:
# TODO: build a pretrained ViT with common.make_vit_base(pretrained=True)
# TODO: finetune for 3 epochs at lr=1e-3 on the 224px loaders
results["vit"] = history[-1][1]
print(f"vit val accuracy: {history[-1][1]:.3f}")

## The leaderboard you just built

In [ ]:
import matplotlib.pyplot as plt
names = list(results.keys())
accs = [results[n] for n in names]
plt.figure(figsize=(7, 4))
bars = plt.bar(names, accs, color="#40E0D0")
plt.ylabel("validation accuracy")
plt.title("Same data, five models")
for b, a in zip(bars, accs):
    plt.text(b.get_x() + b.get_width() / 2, a + 0.01, f"{a:.2f}", ha="center")
plt.ylim(0, 1)
plt.show()

## Bridge to Day 2

The Vision Transformer worked by splitting the image into patches and letting them
attend to each other:

```
   IMAGE                                     SENTENCE
   [patch][patch][patch]  --embed-->         "the"  "cat"  "sat"  --embed-->
        vectors                                    vectors
          |                                          |
       attention  (which patches matter?)         attention  (which words matter?)
          |                                          |
       prediction: DR grade                       prediction: next word
```

That second column is a **Large Language Model**. Same machinery, different input.
Tomorrow we use one to read radiology reports. See you then.

## Stretch -- find a disagreement

If you finished early: find an eye image where the ResNet and the ViT predicted
*different* grades. Show the image and both predictions. What's unusual about it?
Which model would you trust, and why?